# 05 - Queries SQL y Dashboard

En este notebook se ejecutan las consultas SQL que responden las preguntas analíticas del proyecto.

Estas consultas sirven como base para construir el dashboard ejecutivo en Databricks.

In [0]:
%sql
USE CATALOG anemia_ayacucho;
USE SCHEMA gold;

## Pregunta 1
¿Cuál es el total de niños evaluados, niños con anemia y tasa general de anemia?

In [0]:
%sql
SELECT *
FROM vw_kpi_general;

total_evaluados,total_ninos_anemia,tasa_anemia_general,total_registros,total_provincias,total_distritos
60557,17366,28.68,1647,6,62


## Pregunta 2
¿Cómo evolucionó la anemia infantil entre 2023, 2024 y 2025?

In [0]:
%sql
SELECT *
FROM vw_evolucion_anual;

anio,total_evaluados,total_ninos_anemia,tasa_anemia
2023,20371,7356,36.11
2024,20877,6233,29.86
2025,19309,3777,19.56


## Pregunta 3
¿Qué provincias concentran mayor cantidad de niños con anemia?

In [0]:
%sql
SELECT *
FROM vw_anemia_provincia
ORDER BY total_ninos_anemia DESC;

provincia,total_evaluados,total_ninos_anemia,tasa_anemia
Huamanga,33318,11556,34.68
Huanta,12515,2575,20.58
La Mar,9528,1825,19.15
Cangallo,3392,835,24.62
Lucanas,962,380,39.5
Huanca Sancos,842,195,23.16


## Pregunta 4
¿Qué zonas presentan mayor cantidad de niños con anemia?

In [0]:
%sql
SELECT *
FROM vw_anemia_zona
ORDER BY total_ninos_anemia DESC;

zona,tipo_zona,total_evaluados,total_ninos_anemia,tasa_anemia
Zona Norte,Cardinal,6807,2008,29.5
Zona Sur,Cardinal,6782,1984,29.25
Zona Este,Cardinal,6758,1965,29.08
Zona Oeste,Cardinal,6738,1948,28.91
Zona Centro,Central,6730,1929,28.66
Zona Noreste,Intercardinal,6713,1908,28.42
Zona Noroeste,Intercardinal,6687,1886,28.2
Zona Suroeste,Intercardinal,6675,1874,28.07
Zona Sureste,Intercardinal,6667,1864,27.96


## Pregunta 5
¿Qué distritos presentan mayor tasa de anemia y requieren prioridad de análisis?

In [0]:
%sql
SELECT *
FROM vw_ranking_distritos_criticos
ORDER BY tasa_anemia DESC, total_ninos_anemia DESC
LIMIT 15;

provincia,distrito,total_evaluados,total_ninos_anemia,tasa_anemia
Huanta,Santillana,499,290,58.12
Huamanga,Socos,747,434,58.1
Lucanas,Chipao,211,107,50.71
Huamanga,Acocro,1086,550,50.64
Lucanas,Carmen Salcedo,92,44,47.83
Lucanas,Aucara,156,71,45.51
Huanta,Pucacolpa,442,201,45.48
Huamanga,Jesus Nazareno,2620,1185,45.23
Huanta,Iguain,370,163,44.05
Huamanga,Quinua,513,213,41.52


## Pregunta 6
¿Cuál es el ranking anual de provincias según cantidad de niños con anemia?

In [0]:
%sql
SELECT
    t.anio,
    u.provincia,
    SUM(f.nro_evaluados) AS total_evaluados,
    SUM(f.nro_ninos_anemia) AS total_ninos_anemia,
    ROUND((SUM(f.nro_ninos_anemia) / SUM(f.nro_evaluados)) * 100, 2) AS tasa_anemia,
    DENSE_RANK() OVER (
        PARTITION BY t.anio
        ORDER BY SUM(f.nro_ninos_anemia) DESC
    ) AS ranking_anual
FROM fact_anemia f
LEFT JOIN dim_tiempo t
ON f.id_tiempo = t.id_tiempo
LEFT JOIN dim_ubicacion u
ON f.id_ubicacion = u.id_ubicacion
GROUP BY t.anio, u.provincia
ORDER BY t.anio, ranking_anual;

anio,provincia,total_evaluados,total_ninos_anemia,tasa_anemia,ranking_anual
2023,Huamanga,11266,4830,42.87,1
2023,Huanta,4128,1142,27.66,2
2023,La Mar,3258,789,24.22,3
2023,Cangallo,1131,361,31.92,4
2023,Lucanas,310,149,48.06,5
2023,Huanca Sancos,278,85,30.58,6
2024,Huamanga,11555,4120,35.66,1
2024,Huanta,4266,933,21.87,2
2024,La Mar,3289,669,20.34,3
2024,Cangallo,1151,307,26.67,4


## Pregunta 7
¿La tasa de anemia aumentó o disminuyó respecto al año anterior?

In [0]:
%sql
WITH evolucion AS (
    SELECT
        anio,
        total_evaluados,
        total_ninos_anemia,
        tasa_anemia,
        LAG(tasa_anemia) OVER (ORDER BY anio) AS tasa_anio_anterior
    FROM vw_evolucion_anual
)
SELECT
    anio,
    total_evaluados,
    total_ninos_anemia,
    tasa_anemia,
    tasa_anio_anterior,
    ROUND(tasa_anemia - tasa_anio_anterior, 2) AS variacion_puntos_porcentuales,
    CASE
        WHEN tasa_anio_anterior IS NULL THEN 'Sin comparación'
        WHEN tasa_anemia > tasa_anio_anterior THEN 'Aumentó'
        WHEN tasa_anemia < tasa_anio_anterior THEN 'Disminuyó'
        ELSE 'Se mantuvo'
    END AS interpretacion
FROM evolucion;

anio,total_evaluados,total_ninos_anemia,tasa_anemia,tasa_anio_anterior,variacion_puntos_porcentuales,interpretacion
2023,20371,7356,36.11,null,null,Sin comparación
2024,20877,6233,29.86,36.11,-6.25,Disminuyó
2025,19309,3777,19.56,29.86,-10.3,Disminuyó


## Sugerencia de dashboard

Para el dashboard ejecutivo se recomienda usar:

- KPI 1: Total de niños evaluados.
- KPI 2: Total de niños con anemia.
- KPI 3: Tasa general de anemia.
- Gráfico de línea: evolución anual de la tasa de anemia.
- Barras horizontales: provincias con más casos de anemia.
- Barras o dona: casos por zona.
- Tabla: distritos críticos.
- Filtros: año, provincia y zona.

In [0]:
%sql
USE CATALOG anemia_ayacucho;
USE SCHEMA gold;

SHOW TABLES;

database,tableName,isTemporary
gold,dim_tiempo,false
gold,dim_ubicacion,false
gold,dim_zona,false
gold,fact_anemia,false
gold,resumen_dashboard,false
gold,vw_anemia_provincia,false
gold,vw_anemia_zona,false
gold,vw_evolucion_anual,false
gold,vw_kpi_general,false
gold,vw_ranking_distritos_criticos,false
